In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

file_path = r"C:\Users\gerre\OneDrive\Рабочий стол\dataset\dst_no_duplicates.csv"
df = pd.read_csv(file_path, sep=';', encoding='utf-8')

In [ ]:
def extract_age(value):
    try:
        if not isinstance(value, str):
            return np.nan
        parts = value.split(',')
        if len(parts) < 2:
            return np.nan
        age_part = parts[1].strip()  # например "39 лет"
        digits = ''.join(filter(str.isdigit, age_part))
        if digits == '':
            return np.nan
        return int(digits)
    except Exception:
        return np.nan

df['Возраст'] = df['Пол, возраст'].apply(extract_age)
df_age = df.dropna(subset=['Возраст']).copy()

In [ ]:
df_age['log_age'] = np.log(df_age['Возраст'] + 1)

log_mean = df_age['log_age'].mean()
log_std = df_age['log_age'].std()

lower_bound = log_mean - 3*log_std
upper_bound = log_mean + 3*log_std

plt.figure(figsize=(10, 6))
histplot = sns.histplot(df_age['log_age'], bins=30, kde=False, color='skyblue')

# Линии среднего и 3 сигм
plt.axvline(log_mean, color='k', lw=2, label='Среднее')
plt.axvline(lower_bound, color='r', ls='--', lw=2, label='Среднее - 3σ')
plt.axvline(upper_bound, color='g', ls='--', lw=2, label='Среднее + 3σ')

plt.title('Распределение логарифма возраста (log(Возраст + 1))')
plt.xlabel('log(Возраст + 1)')
plt.ylabel('Частота')
plt.legend()
plt.show()

График показывает распределение возраста в логарифмическом масштабе.  
Распределение асимметрично вправо — хвост скошен в сторону больших значений возраста.  
Это типично для возрастных показателей, где есть редкие, но очень большие значения (преклонный возраст).  
Линии показывают среднее значение и границы интервала метода трёх сигм (±3σ).

In [ ]:
df_age['z_score'] = (df_age['log_age'] - log_mean) / log_std

lower_limit = -3
upper_limit = 4  # расширяем правую границу до 4 сигм

outliers = df_age[(df_age['z_score'] < lower_limit) | (df_age['z_score'] > upper_limit)]

print("Выбросы по возрасту (log-возраст за пределами [-3σ, +4σ]):")
display(outliers[['Пол, возраст', 'Возраст', 'log_age', 'z_score']].sort_values(by='z_score'))

num_outliers = outliers.shape[0]
print(f"\nКоличество выбросов, найденных с помощью метода z-отклонений: {num_outliers}")

if num_outliers > 0:
    print("Минимальный возраст выбросов:", outliers['Возраст'].min())
    print("Максимальный возраст выбросов:", outliers['Возраст'].max())
else:
    print("Выбросы не найдены.")

Мы обнаружили **{num_outliers}** потенциальных выбросов по возрасту с помощью метода z-отклонений с расширением правой границы до 4 сигм.  

Выбросы — это в основном резюме людей с очень высоким возрастом, неподходящим для активного поиска работы.  
Минимальный и максимальный возраст среди выбросов позволяют понять диапазон этих исключительных значений.  

Удаление таких выбросов позволяет сделать анализ данных более корректным и избежать влияния аномальных пожилых возрастов.